In [7]:
import tensorflow as tf
from tensorflow.keras import layers
import tensorboard as tb
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

print("TensorFlow version: ", tf.__version__)

df = pd.read_excel('electricity_usage_modified.xlsx')


#for col in df.columns:
#    if pd.api.types.is_datetime64_any_dtype(df[col]):
#        df[col] = df[col].astype(np.int64) // 10**9  # Convert to seconds (integer)

print(df.head())

TensorFlow version:  2.20.0
             Datetime       date  year  month  day_of_month  day_of_week  \
0 1998-12-31 01:00:00 1998-12-31  1998     12            31            5   
1 1998-12-31 02:00:00 1998-12-31  1998     12            31            5   
2 1998-12-31 03:00:00 1998-12-31  1998     12            31            5   
3 1998-12-31 04:00:00 1998-12-31  1998     12            31            5   
4 1998-12-31 05:00:00 1998-12-31  1998     12            31            5   

   day_of_year  semester  quarter  hour_of_day  total_usage  
0          365         2        4            1        29309  
1          365         2        4            2        28236  
2          365         2        4            3        27692  
3          365         2        4            4        27596  
4          365         2        4            5        27888  


In [8]:
cols_to_drop = ['Datetime', 'date', 'total_usage']


y = df['total_usage']
x = df.drop(columns = cols_to_drop)

x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size= 0.2,
    random_state=42
)

print(f"Training set size: {len(x_train)} samples")
print(f"Testing set size: {len(x_test)} samples")

Training set size: 142609 samples
Testing set size: 35653 samples


In [9]:
train_dataset = tf.data.Dataset.from_tensor_slices((x_train.values, y_train.values))
test_dataset = tf.data.Dataset.from_tensor_slices((x_test.values, y_test.values))

BUFFER_SIZE = 1000
BATCH_SIZE = 32

train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [10]:
input_shape = x_train.shape[1]
normalizer = layers.Normalization(axis=-1)
normalizer.adapt(x_train.values)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape = (input_shape,)),
    normalizer,
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),

    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mean_squared_error',
    metrics=['mean_absolute_error']
)

In [11]:
history = model.fit(
    train_dataset,
    epochs=50,
    validation_data=test_dataset
)

Epoch 1/50
4457/4457 ━━━━━━━━━━━━━━━━━━━━ 3s 468us/step - loss: 633071680.0000 - mean_absolute_error: 17679.1953 - val_loss: 255517232.0000 - val_mean_absolute_error: 12764.7021
Epoch 2/50
4457/4457 ━━━━━━━━━━━━━━━━━━━━ 2s 447us/step - loss: 258270800.0000 - mean_absolute_error: 12787.8145 - val_loss: 247384768.0000 - val_mean_absolute_error: 12461.5400
Epoch 3/50
4457/4457 ━━━━━━━━━━━━━━━━━━━━ 2s 451us/step - loss: 250388144.0000 - mean_absolute_error: 12588.6855 - val_loss: 235787936.0000 - val_mean_absolute_error: 12176.1025
Epoch 4/50
4457/4457 ━━━━━━━━━━━━━━━━━━━━ 2s 442us/step - loss: 238901872.0000 - mean_absolute_error: 12314.0059 - val_loss: 227585232.0000 - val_mean_absolute_error: 11985.5098
Epoch 5/50
4457/4457 ━━━━━━━━━━━━━━━━━━━━ 2s 450us/step - loss: 234868032.0000 - mean_absolute_error: 12217.5908 - val_loss: 226111904.0000 - val_mean_absolute_error: 11959.7617
Epoch 6/50
4457/4457 ━━━━━━━━━━━━━━━━━━━━ 2s 446us/step - loss: 235149616.0000 - mean_absolute_error: 12227.53

KeyboardInterrupt: 